# New College Dataset pipeline

This notebook replaces the synthetic sensor import used in `09_*.ipynb` with real New College IMU and LiDAR streams. Keep downstream processing cells aligned with the original notebook.

In [ ]:
# Configuration: keep all notebook knobs here.
from pathlib import Path

DATASET_ROOT = Path('/mnt/d/Downloads/MobRobLab/NewCollegeDataset/01_short_experiment')
TARGET_IMU_FREQUENCY_HZ = 100.0
MAX_IMU_FILES = None

LIDAR_VOXEL_SIZE_M = 0.5
LIDAR_MAX_CORRESPONDENCE_DISTANCE_M = 1.5
LIDAR_MAX_ITERATIONS = 50
LIDAR_MAX_SCANS = None
LIDAR_SCAN_PERIOD_S = None
EXTRACT_LIDAR_IF_NEEDED = True

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_ROOT = PROJECT_ROOT / 'src'
NOTEBOOKS_ROOT = PROJECT_ROOT / 'notebooks'

In [ ]:
import sys

import nbformat
import numpy as np

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from new_college_dataset import load_imus, load_lidar_relative_poses
from new_college_dataset.lidar import load_lidar_data_csv
from transform import se3_to_angvels, se3_to_velocities

## Original notebook reference

In [ ]:
source_notebooks = sorted(NOTEBOOKS_ROOT.glob('09_*.ipynb'))
if len(source_notebooks) != 1:
    raise RuntimeError(f'Expected exactly one 09_*.ipynb notebook, found {len(source_notebooks)}: {source_notebooks}')

SOURCE_NOTEBOOK = source_notebooks[0]
source_nb = nbformat.read(SOURCE_NOTEBOOK, as_version=4)
print('Using original pipeline notebook:', SOURCE_NOTEBOOK)
print('Original notebook cells:', len(source_nb.cells))

## Real New College sensor import

These variables intentionally use simple names so the downstream cells from `09_*.ipynb` can bind to real sensor streams instead of simulated streams.

In [ ]:
imu_streams = load_imus(
    DATASET_ROOT,
    target_frequency_hz=TARGET_IMU_FREQUENCY_HZ,
    max_files=MAX_IMU_FILES,
)
primary_imu_name = next(iter(imu_streams))
primary_imu = imu_streams[primary_imu_name]

lidar_data = load_lidar_data_csv(
    '~/Skoltech/phd_proposal/data/NewCollegeDataset/lidar_odometry.csv',
)

velocity_timestamps = lidar_data.scan_timestamps_s if lidar_data.scan_timestamps_s is not None else lidar_data.timestamps_s
lidar_angvel_radps = se3_to_angvels(lidar_data.relative_poses_se3, velocity_timestamps)
lidar_velocity_mps = se3_to_velocities(lidar_data.relative_poses_se3, velocity_timestamps)

print('Primary IMU:', primary_imu_name, primary_imu.timestamps_s.shape)
print('LiDAR relative poses:', lidar_data.relative_poses_se3.shape)

## Pipeline continuation

Continue here with the downstream processing cells from `09_*.ipynb`, replacing only references to simulated sensor variables with `primary_imu`, `imu_streams`, `lidar_data`, `lidar_angvel_radps`, and `lidar_velocity_mps` as needed.

## 1. Wrapped Pipeline Imports

These imports mirror notebook 09. The only extra step in notebook 11 is converting the already imported New College streams into the dataset interface used by the observability package.


In [ ]:
from IPython.display import HTML, Image, display

from calib_observability.backend import estimate_poses_dummy
from calib_observability.diagnostics import DEFAULT_PRACTICAL_RANK_POLICY
from calib_observability.factor_observability import SUPPORTED_CALIBRATION_VARIABLES
from calib_observability.scaling import ParameterScales
from calib_observability.simulation import reframe_dataset_to_fixed_extrinsic
from calib_observability.types import AccelerometerOptions, JacobianOptions
from calib_observability.workflows import (
    build_dataset_from_imported_sensor_streams,
    plot_local_crlb_accuracy,
    plot_observability_over_time,
    plot_rover_dataset_overview,
    run_rolling_observability_analysis,
    save_simple_accelerometer_dashboard,
)


## 2. Observability Header Configuration

All tunable analysis variables are kept here, following notebook 09 names where possible. The imported-data bridge uses nominal calibration values as the linearization point; update these once real calibration priors are available.


In [ ]:
OUT = PROJECT_ROOT / 'outputs' / 'new_college_observability_pipeline'
OUT.mkdir(parents=True, exist_ok=True)

FIXED_EXTRINSIC = 'T_B_L'
WINDOW_LENGTH = 5.0
WINDOW_STEP = 1.0
USE_SPARSE = False
NORMALIZATION = 'physical_then_column'
DISPLAY_VARIABLES = tuple(SUPPORTED_CALIBRATION_VARIABLES)
MAX_DISPLAY_ROWS = 80
MAX_DISPLAY_COLS = 40
PARAMETER_SCALES = ParameterScales()
PRACTICAL_RANK_POLICY = DEFAULT_PRACTICAL_RANK_POLICY
JACOBIAN_OPTIONS = JacobianOptions(method='analytic')
COORDINATE_NULL_FRACTION_TOLERANCE = 1e-6

# Imported-data noise/linearization assumptions. These are not measured truths;
# they define the local whitening and calibration point for observability.
IMPORTED_GYRO_NOISE_STD_RADPS = 0.01
IMPORTED_ACCEL_NOISE_STD_MPS2 = 0.10
IMPORTED_LIDAR_POSE_NOISE_STD = np.array([0.02, 0.02, 0.02, 0.10, 0.10, 0.10], dtype=float)
T_B_I_INITIAL_TANGENT = np.zeros(6, dtype=float)
T_B_L_INITIAL_TANGENT = np.zeros(6, dtype=float)
GYRO_BIAS_INITIAL = np.zeros(3, dtype=float)
TAU_I_INITIAL = 0.0
TAU_L_INITIAL = 0.0
INVERT_IMPORTED_LIDAR_MEASUREMENTS = False

# Timing diagnostics use an estimated LiDAR rate from scan timestamps.
if lidar_data.scan_timestamps_s is not None and len(lidar_data.scan_timestamps_s) > 1:
    LIDAR_RATE_HZ = 1.0 / float(np.median(np.diff(lidar_data.scan_timestamps_s)))
elif len(lidar_data.timestamps_s) > 1:
    LIDAR_RATE_HZ = 1.0 / float(np.median(np.diff(lidar_data.timestamps_s)))
elif LIDAR_SCAN_PERIOD_S is not None:
    LIDAR_RATE_HZ = 1.0 / float(LIDAR_SCAN_PERIOD_S)
else:
    LIDAR_RATE_HZ = 5.0
TAU_TARGET_FRAMES = 1.0
TAU_TARGET_STD_SECONDS = TAU_TARGET_FRAMES / LIDAR_RATE_HZ

SIMPLE_ACCELEROMETER_OPTIONS = AccelerometerOptions(
    mode='simple',
    factor_rate_hz=LIDAR_RATE_HZ,
    support_half_width_seconds=0.2,
    gravity_norm_tolerance_m_s2=0.75,
    low_dynamic_gyro_threshold_rad_s=0.35,
    require_low_dynamic_gate=True,
    measurement_std_m_s2=IMPORTED_ACCEL_NOISE_STD_MPS2,
    save_factor_terms=True,
)
ANIMATION_HTML = OUT / 'new_college_simple_accelerometer_dashboard.html'
SAVE_ANIMATION_MP4 = False
ANIMATION_MP4 = OUT / 'new_college_simple_accelerometer_dashboard.mp4'
ANIMATION_INTERVAL_MS = 300
ANIMATION_TRAJECTORY_SAMPLES = 700

print({
    'output': str(OUT),
    'window_length': WINDOW_LENGTH,
    'window_step': WINDOW_STEP,
    'lidar_rate_hz_estimate': LIDAR_RATE_HZ,
    'display_variables': DISPLAY_VARIABLES,
})


## 3. Build Observability Dataset From Imported Sensors

This cell is the replacement for notebook 09 simulation. It reconstructs a discrete body trajectory from cumulative LiDAR odometry, aligns IMU samples to the LiDAR time span, and creates the same `dataset` plus `pose_provider` variables used downstream.


In [ ]:
def lidar_scan_timestamps_for_pipeline(lidar_data):
    if lidar_data.scan_timestamps_s is not None:
        return np.asarray(lidar_data.scan_timestamps_s, dtype=float)
    midpoint_times = np.asarray(lidar_data.timestamps_s, dtype=float)
    if midpoint_times.size == 0:
        raise ValueError('LiDAR data has no timestamps')
    if midpoint_times.size == 1:
        if LIDAR_SCAN_PERIOD_S is None:
            raise ValueError('Set LIDAR_SCAN_PERIOD_S when only one LiDAR midpoint timestamp is available')
        half_period = 0.5 * float(LIDAR_SCAN_PERIOD_S)
        return np.array([midpoint_times[0] - half_period, midpoint_times[0] + half_period], dtype=float)
    boundaries = np.empty(midpoint_times.size + 1, dtype=float)
    boundaries[1:-1] = 0.5 * (midpoint_times[:-1] + midpoint_times[1:])
    boundaries[0] = midpoint_times[0] - (boundaries[1] - midpoint_times[0])
    boundaries[-1] = midpoint_times[-1] + (midpoint_times[-1] - boundaries[-2])
    return boundaries

lidar_scan_timestamps_s = lidar_scan_timestamps_for_pipeline(lidar_data)
raw_dataset = build_dataset_from_imported_sensor_streams(
    imu_timestamps=primary_imu.timestamps_s,
    gyroscope=primary_imu.gyro_radps,
    accelerometer=primary_imu.accel_mps2,
    lidar_scan_timestamps=lidar_scan_timestamps_s,
    lidar_relative_poses=lidar_data.relative_poses_se3,
    T_B_I_initial_tangent=T_B_I_INITIAL_TANGENT,
    T_B_L_initial_tangent=T_B_L_INITIAL_TANGENT,
    gyro_bias_initial=GYRO_BIAS_INITIAL,
    tau_I_initial=TAU_I_INITIAL,
    tau_L_initial=TAU_L_INITIAL,
    gyro_noise_std=IMPORTED_GYRO_NOISE_STD_RADPS,
    accel_noise_std=IMPORTED_ACCEL_NOISE_STD_MPS2,
    lidar_pose_noise_std=IMPORTED_LIDAR_POSE_NOISE_STD,
    invert_lidar_measurements=INVERT_IMPORTED_LIDAR_MEASUREMENTS,
)
dataset = reframe_dataset_to_fixed_extrinsic(raw_dataset, FIXED_EXTRINSIC)
pose_provider = estimate_poses_dummy(dataset)

print({
    'dataset_start': dataset.start_time,
    'dataset_end': dataset.end_time,
    'imu_samples': len(dataset.imu.sensor_timestamps),
    'lidar_measurements': len(dataset.lidar.measurements),
    'trajectory_mode': dataset.trajectory.mode,
})


## 4. Imported Data Overview

This is the same overview section as notebook 09, now driven by imported New College streams. The trajectory is the cumulative LiDAR odometry path used as the current pose estimate.


In [ ]:
overview_paths = plot_rover_dataset_overview(dataset, OUT, trajectory_samples=900)
for name, path in overview_paths.items():
    print(name, path)
    display(Image(filename=str(path)))


## 5. Rolling Observability Analyzer

Run the canonical rolling-window observability analyzer on the imported dataset. This first pass keeps the accelerometer disabled, matching notebook 09's base analysis; the final dashboard cell enables simple accelerometer factors.


In [ ]:
analysis_series = run_rolling_observability_analysis(
    dataset,
    pose_provider,
    window_duration=WINDOW_LENGTH,
    window_step=WINDOW_STEP,
    fixed_extrinsic=FIXED_EXTRINSIC,
    practical_rank_policy=PRACTICAL_RANK_POLICY,
    parameter_scales=PARAMETER_SCALES,
    tau_target_std_seconds=TAU_TARGET_STD_SECONDS,
    jacobian_options=JACOBIAN_OPTIONS,
    accelerometer_options=None,
    use_sparse=USE_SPARSE,
    display_variables=DISPLAY_VARIABLES,
    normalization=NORMALIZATION,
    max_display_rows=MAX_DISPLAY_ROWS,
    max_display_cols=MAX_DISPLAY_COLS,
    lidar_rate_hz=LIDAR_RATE_HZ,
    coordinate_null_fraction_tolerance=COORDINATE_NULL_FRACTION_TOLERANCE,
)
valid_count = sum(snapshot.is_valid for snapshot in analysis_series.snapshots)
print({'snapshots': len(analysis_series.snapshots), 'valid_snapshots': valid_count})


## 6. Observability Results Over Time

These plots are generated by the wrapped notebook-09 logic: practical ranks, practical condition numbers, and factor counts.


In [ ]:
observability_paths = plot_observability_over_time(
    analysis_series,
    OUT,
    display_variables=DISPLAY_VARIABLES,
)
for name, path in observability_paths.items():
    print(name, path)
    display(Image(filename=str(path)))


## 7. Local CRLB-like Accuracy Diagnostics

These are local CRLB-like standard-deviation bounds from the physical, unnormalized projected matrices. For imported data, interpret the magnitudes together with the assumed noise values and the LiDAR-derived pose trajectory quality.


In [ ]:
accuracy_paths = plot_local_crlb_accuracy(
    analysis_series,
    OUT,
    display_variables=DISPLAY_VARIABLES,
)
for name, path in accuracy_paths.items():
    print(name, path)
    display(Image(filename=str(path)))


## 8. Quasi-Realtime Dashboard With Simple Accelerometer Mode

This reproduces the notebook-08 visualization style for the imported New College dataset. The simple accelerometer factor uses low-dynamic gravity-alignment gates and shares `T_B_I` and `tau_I` with the gyroscope.


In [ ]:
simple_series, animation_path = save_simple_accelerometer_dashboard(
    dataset,
    pose_provider,
    ANIMATION_HTML,
    window_duration=WINDOW_LENGTH,
    window_step=WINDOW_STEP,
    accelerometer_options=SIMPLE_ACCELEROMETER_OPTIONS,
    fixed_extrinsic=FIXED_EXTRINSIC,
    practical_rank_policy=PRACTICAL_RANK_POLICY,
    parameter_scales=PARAMETER_SCALES,
    tau_target_std_seconds=TAU_TARGET_STD_SECONDS,
    jacobian_options=JACOBIAN_OPTIONS,
    use_sparse=USE_SPARSE,
    display_variables=DISPLAY_VARIABLES,
    normalization=NORMALIZATION,
    max_display_rows=MAX_DISPLAY_ROWS,
    max_display_cols=MAX_DISPLAY_COLS,
    lidar_rate_hz=LIDAR_RATE_HZ,
    coordinate_null_fraction_tolerance=COORDINATE_NULL_FRACTION_TOLERANCE,
    trajectory_samples=ANIMATION_TRAJECTORY_SAMPLES,
    interval_ms=ANIMATION_INTERVAL_MS,
    output_mp4=ANIMATION_MP4 if SAVE_ANIMATION_MP4 else None,
)
print(animation_path)
display(HTML(animation_path.read_text(encoding='utf-8')))


## 9. Output Summary

The dictionary below collects the main output artifacts from the imported-data observability run.


In [ ]:
all_paths = {
    'overview': overview_paths,
    'observability': observability_paths,
    'accuracy': accuracy_paths,
    'simple_accelerometer_animation': animation_path,
}
all_paths
